### Install packages

In [1]:
%pip install -q networkx pandas matplotlib python-louvain pyvis affiliation-builder  # Install quietly

Note: you may need to restart the kernel to use updated packages.


### Import dependencies

In [2]:
# Core libraries
import json
import logging
from pathlib import Path

# Network analysis
import networkx as nx
from networkx.algorithms import bipartite
from community import community_louvain

# Data manipulation
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# affiliation-builder
from affiliation_builder import build

### Configure logging

In [3]:
logging.getLogger('affiliation_builder').setLevel(logging.INFO)  # Display console feedback
logging.getLogger('affiliation_builder').addHandler(logging.StreamHandler())

### Load JSON and build bipartite network

In [4]:
G_bipartite = build(
    json_path="../examples/amp-events.json",
    node_set_0_key="listEvent",
    node_set_1_keys=["listPerson", "org"],
    identifier_key="xml:id",
    node_set_1_identifier_key="sameAs"
)

Affiliation network builder started
JSON source: ../examples/amp-events.json
Node-set-0 key: 'listEvent'
Node-set-1 keys: ['listPerson', 'org']
Identifier key: 'xml:id'
Node-set-1 identifier key: 'sameAs'
Local file path detected: ../examples/amp-events.json
Local JSON file successfully loaded
Wrapped object format with 105 items in 'listEvent' detected
NetworkX graph initialized
Bipartite network construction complete
Total edges (affiliations): 245
Total nodes: 188
Node set 0 (listEvent): 105 nodes
Node set 1 (['listPerson', 'org']): 83 nodes
✓ Graph structure is valid bipartite


### Extract node sets

In [5]:
events = {n for n, d in G_bipartite.nodes(data=True) if d['bipartite'] == 0}  # Create set of node IDs in node-set 0
participants = {n for n, d in G_bipartite.nodes(data=True) if d['bipartite'] == 1}

### Filter bipartite graph

#### Define filter

In [6]:
# Filter by event date

def filter_by_date(G_bipartite, start_date, end_date):
    """
    Filter bipartite graph to events within a date range.
    Uses notBefore-iso attribute for filtering.
    
    Parameters
    ----------
    G_bipartite : networkx.Graph
        Bipartite event-participant graph
    start_date : str
        Start date in ISO format (e.g., '1960-01-01')
    end_date : str
        End date in ISO format (e.g., '1970-12-31')
    
    Returns
    -------
    networkx.Graph
        Filtered bipartite graph
    """
    filtered_events = []  # List to collect relevant event node IDs

    for node, data in G_bipartite.nodes(data=True):  # Loop through all nodes with their IDs and attributes
        if data.get('bipartite') == 0:  # Get events
            event_date = data.get('notBefore-iso', '')[:10]  # Get event start date attribute
            if event_date and start_date <= event_date <= end_date:  # Check event start date is in range
                filtered_events.append(node)  # Add event node ID
    
    filtered_participants = set()  # Set to collect relevant participant node IDs

    for event in filtered_events:
        filtered_participants.update(G_bipartite.neighbors(event))  # Add participant node IDs
    
    filtered_nodes = set(filtered_events) | filtered_participants  # Combine relevant events and participants in single set
    
    return G_bipartite.subgraph(filtered_nodes).copy()  # Return independent filtered graph


#### Apply filters

In [7]:
G_filtered_date = filter_by_date(G_bipartite, '1957-01-01', '1973-09-29')  # Filter by event date

# Update node sets
events = {n for n, d in G_filtered_date.nodes(data=True) if d['bipartite'] == 0}
participants = {n for n, d in G_filtered_date.nodes(data=True) if d['bipartite'] == 1}

### Unipartite projection

In [8]:
G = bipartite.weighted_projected_graph(G_filtered_date, participants)  # Create participant network with weighted edges

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

print("\nStrongest connections:")
edges_by_weight = sorted(G.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)  # From node-node-attributes tuples, create list of edges sorted by weight, highest first
for u, v, d in edges_by_weight[:5]:  # Unpack
    print(f"  {u} — {v}: {d['weight']} shared events")

Nodes: 62
Edges: 164

Strongest connections:
  amp_person_2 — amp_person_1: 34 shared events
  amp_person_4 — amp_person_1: 11 shared events
  amp_person_4 — amp_person_2: 7 shared events
  amp_organization_9 — amp_person_1: 3 shared events
  amp_person_137 — amp_person_1: 2 shared events


### Extract ego network

In [9]:
ego = 'amp_person_1'  # Define ego

G_ego = nx.ego_graph(G, ego)  # Extract network with ego and directly connected alters, including alter-alter edges

print(f"Alters: {G_ego.number_of_nodes() - 1}")  # Subtract ego
print(f"Edges: {G_ego.number_of_edges()}")

print("\nStrongest connections:")
edges_by_weight = sorted(G_ego.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)
for u, v, d in edges_by_weight[:5]:
    print(f"  {u} — {v}: {d['weight']} shared events")

Alters: 45
Edges: 129

Strongest connections:
  amp_person_2 — amp_person_1: 34 shared events
  amp_person_4 — amp_person_1: 11 shared events
  amp_person_4 — amp_person_2: 7 shared events
  amp_organization_9 — amp_person_1: 3 shared events
  amp_person_137 — amp_person_1: 2 shared events
